## Creating Agentic Workflows in LlamaIndex

### Creating Workflows

In [1]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step, Context
from llama_index.core.workflow import Event
from llama_index.utils.workflow import draw_all_possible_flows
import random

In [2]:
# Define a custom workflow by inheriting from the Workflow class
class MyWorkflow(Workflow):
    # Define a workflow step that receives the starting event
    @ step
    async def my_step(self, ev: StartEvent) -> StopEvent:
        # Perform the workflow task here
        return StopEvent(result="Hello, world!")

In [3]:
# Create an instance of the workflow with a 10-second timeout
# and disable verbose logging
w = MyWorkflow(timeout=10, verbose=False)
result = await w.run()

In [4]:
print(result)

Hello, world!


In [5]:
# Define a custom event to pass information between workflow steps
class ProcessingEvent(Event):
    intermediate_result: str

In [6]:
# Define a workflow with multiple processing steps
class MultiStepWorkflow(Workflow):
    @ step 
    async def step_one(self, ev: StartEvent) -> ProcessingEvent:
        # Create an event containing the intermediate result
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @ step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result to create the final result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)

In [7]:
# Create the workflow with a 10-second timeout and disable verbose output
w = MultiStepWorkflow(timeout=10, verbose=False)
result = await w.run()

In [8]:
print(result)

Finished processing: Step 1 complete


In [9]:
# Event used to pass the intermediate result between steps
class ProcessingEvent(Event):
    intermediate_result: str

In [10]:
# Event used to loop back to the first step
class LoopEvent(Event):
    loop_output: str

In [11]:
# Define a workflow that can loop between steps
class MultiStepWorkflow(Workflow):

    @step
    async def step_one(
        self, ev: StartEvent | LoopEvent
    ) -> ProcessingEvent | LoopEvent:

        # Randomly decide whether the step succeeds or loops
        if random.randint(0, 1) == 0:
            print("Bad thing happened")

            # Send the workflow back to step one
            return LoopEvent(loop_output="Back to step one.")

        else:
            print("Good thing happened")

            # Continue to the second step with the result
            return ProcessingEvent(
                intermediate_result="First step complete."
            )

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:

        # Use the intermediate result to create the final result
        final_result = f"Finished processing: {ev.intermediate_result}"

        # Stop the workflow and return the final result
        return StopEvent(result=final_result)

In [12]:
# Create the workflow with verbose output disabled
w = MultiStepWorkflow(verbose=False)
result = await w.run()

Good thing happened


In [13]:
print(result)

Finished processing: First step complete.


In [14]:
draw_all_possible_flows(w, "flow.html")

flow.html


### Automating Workflows with Multi-Agent Workflows

In [15]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

In [16]:
# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [ ]:
# Initialize the Hugging Face language model
llm = HuggingFaceInferenceAPI(
    model_name="meta-llama/Llama-3.2-3B-Instruct",
    token="")

In [18]:
# Create an agent specialized in multiplication
multiply_agent = ReActAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="A helpful assistant that can use a tool to multiply numbers.",
    tools=[multiply],
    llm=llm,
)

# Create an agent specialized in addition
addition_agent = ReActAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="A helpful assistant that can use a tool to add numbers.",
    tools=[add],
    llm=llm,
)

In [19]:
# Create an AgentWorkflow containing both specialized agents
workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
)

In [20]:
# Run the system
response = await workflow.run(user_msg="Can you add 5 and 3?")

In [21]:
print(response)

The sum of 5 and 3 is 8.
